In [3]:
import subprocess

import json
import os
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

from evals_local.loaders import (
    load_predictions,
    load_screening_outputs,
    load_screening_predictions,
    parse_output_record,
)
from evals_local.normalizers import normalize_screening_label

In [4]:
# config

# Path to source data from WOS to get additional fields if necessary (e.g. publication year, title, abstract)
source_data = "data/partitions/wos_all.csv"

# Global path for all data coding outputs
batch_outputs_folder = "labelling/artifacts/batch_outputs"

# Path for all partitions that are screened 
# (Raw = outputs, curated = expanded coding responses, screened = final screened paritions)
screened_folder = "data/partitions-screened"

# Path to save all eligble records in batch format
batched_folder = 'data/partitions-screened-batched'
batched_folder_limit= 10000

# Path to save all data coding outputs
coded_folder= "data/partitions-coded"


### 1. Screening Task

This involves

1. Running all data split into 234 partitions of 10,000 batch size

2. Processing outputs into /data/partitions-screened in a raw, curated, serving format. Where serving layer contains filtered eligible records

3. Create new batches of max 10,000 size into data/partitions-screened-batch based on aggregation of serving data in partitions-screened. Intenet to compact many partitions into one as screening excludes many records

#### Run partitions 1-234

In [ ]:
for x in range(23, 50):
    cmd = f"python labelling/orchestrator.py data/partitions/{x} p{x}-l0-250226-01 --task screening"
    print(f"\n{'='*60}")
    print(f"Running partition {x}: {cmd}")
    print(f"{'='*60}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(f"STDERR: {result.stderr}")
    if result.returncode != 0:
        print(f"FAILED with return code {result.returncode}")
        break
    print(f"Partition {x} complete.")

#### Add ran partitions to /data/paritions-screened

Input: Add `partition_range` to create records for

For partitions 'p' ran, create their raw, curated, serving records at the screened records path.

/serving contains screened (Eligible) records for each partition

In [36]:
# load source data once for the serving join
_wos_cols = ["UT (Unique WOS ID)", "Article Title", "Abstract", "Publication Year"]
wos_df = pd.read_csv(source_data, usecols=_wos_cols)

for i in partition_range:
    run_name = f"p{i}-{run_suffix}"
    src_dir = Path(batch_outputs_folder) / run_name
    if not src_dir.exists():
        print(f"⚠ Missing: {src_dir}, skipping")
        continue

    # create per-partition folder structure
    part_dir = Path(screened_folder) / str(i)
    raw_dir = part_dir / "raw"
    curated_dir = part_dir / "curated"
    serving_dir = part_dir / "serving"
    for d in [raw_dir, curated_dir, serving_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # ── RAW: concat all jsonl files as-is ─────────────────────────────
    raw_rows = []
    for fp in sorted(src_dir.glob("*.jsonl")):
        with open(fp) as f:
            for line in f:
                rec = json.loads(line)
                rec["_source_run"] = run_name
                rec["_source_file"] = fp.name
                raw_rows.append(rec)

    raw_df = pd.DataFrame(raw_rows)
    raw_path = raw_dir / "screening_raw.csv"
    raw_df.to_csv(raw_path, index=False)

    raw_meta = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "description": "Raw batch output JSONL records",
        "partition": i,
        "run": run_name,
        "num_records": len(raw_df),
        "columns": list(raw_df.columns),
    }
    (raw_dir / "metadata.json").write_text(json.dumps(raw_meta, indent=2))

    # ── CURATED: parse, flatten steps, compute eligibility ────────────
    curated_rows = []
    for fp in sorted(src_dir.glob("*.jsonl")):
        with open(fp) as f:
            for line in f:
                rec = json.loads(line)
                parsed = parse_output_record(rec)
                parsed["_source_run"] = run_name
                curated_rows.append(parsed)

    curated_df = pd.DataFrame(curated_rows)

    for col in ["step_0_label", "step_1_label", "step_2_label", "step_3_label"]:
        if col in curated_df.columns:
            curated_df[col] = pd.to_numeric(curated_df[col], errors="coerce").fillna(0).astype(int)
        else:
            curated_df[col] = 0

    curated_df["label_1_3"] = curated_df[["step_1_label", "step_2_label", "step_3_label"]].sum(axis=1)
    curated_df["pred_screening"] = curated_df.apply(
        lambda r: '["ELIGIBLE"]' if (r["step_0_label"] == 0 and r["label_1_3"] == 3) else '["NOT_ELIGIBLE"]',
        axis=1,
    )
    curated_df["pred_norm"] = curated_df["pred_screening"].apply(normalize_screening_label)

    if "created_at" in curated_df.columns:
        curated_df["created_at"] = pd.to_datetime(curated_df["created_at"], unit="s", utc=True, errors="coerce")

    curated_path = curated_dir / "screening_curated.csv"
    curated_df.to_csv(curated_path, index=False)

    eligible_count = int((curated_df["pred_norm"] == "ELIGIBLE").sum())
    not_eligible_count = int((curated_df["pred_norm"] == "NOT_ELIGIBLE").sum())

    curated_meta = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "description": "Parsed & flattened screening outputs with eligibility prediction",
        "partition": i,
        "run": run_name,
        "num_records": len(curated_df),
        "eligible": eligible_count,
        "not_eligible": not_eligible_count,
        "columns": list(curated_df.columns),
        "eligibility_logic": "ELIGIBLE iff step_0_label==0 AND step_1+step_2+step_3==3",
    }
    (curated_dir / "metadata.json").write_text(json.dumps(curated_meta, indent=2))

    # ── SERVING: only ELIGIBLE records + enriched from source data ────
    serving_df = curated_df[curated_df["pred_norm"] == "ELIGIBLE"].copy()
    n_before = len(serving_df)

    serving_df = serving_df.merge(
        wos_df,
        left_on="custom_id",
        right_on="UT (Unique WOS ID)",
        how="left",
    )
    
    assert len(serving_df) == n_before, f"p{i}: row count changed after join ({n_before} → {len(serving_df)})"
    unmatched = serving_df["Article Title"].isna().sum()
    if unmatched:
        print(f"  ⚠ p{i}: {unmatched} records did not match source_data")

    serving_path = serving_dir / "screening_eligible.csv"
    serving_df.to_csv(serving_path, index=False)

    serving_meta = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "description": "ELIGIBLE records enriched with Article Title, Abstract, Publication Year from source_data",
        "partition": i,
        "run": run_name,
        "num_records": len(serving_df),
        "filter": "pred_norm == 'ELIGIBLE'",
        "enriched_from": source_data,
        "enriched_fields": ["Article Title", "Abstract", "Publication Year"],
        "columns": list(serving_df.columns),
    }
    (serving_dir / "metadata.json").write_text(json.dumps(serving_meta, indent=2))

    pct = len(serving_df) / len(curated_df) * 100 if len(curated_df) else 0
    print(f"p{i}: raw={len(raw_df)}  curated={len(curated_df)}  serving={len(serving_df)} ({pct:.1f}% eligible)")

p1: raw=10000  curated=10000  serving=1244 (12.4% eligible)
p2: raw=10000  curated=10000  serving=1475 (14.8% eligible)
p3: raw=10000  curated=10000  serving=1339 (13.4% eligible)
p4: raw=10000  curated=10000  serving=1266 (12.7% eligible)
p5: raw=10000  curated=10000  serving=1322 (13.2% eligible)
p6: raw=10000  curated=10000  serving=1274 (12.7% eligible)
p7: raw=10000  curated=10000  serving=1330 (13.3% eligible)
p8: raw=10000  curated=10000  serving=1364 (13.6% eligible)
p9: raw=10000  curated=10000  serving=1313 (13.1% eligible)
p10: raw=10000  curated=10000  serving=1360 (13.6% eligible)
p11: raw=10000  curated=10000  serving=1402 (14.0% eligible)
p12: raw=10000  curated=10000  serving=1391 (13.9% eligible)
p13: raw=10000  curated=10000  serving=1243 (12.4% eligible)
p14: raw=10000  curated=10000  serving=1396 (14.0% eligible)
p15: raw=10000  curated=10000  serving=1338 (13.4% eligible)
p16: raw=10000  curated=10000  serving=1465 (14.6% eligible)
p17: raw=10000  curated=10000  se

#### Create /data/partitions-screened-batched to create batches to apply data coding

Input: `batch_groups` is a list of tuple that will group screen recrods into batches of a max size 

Screened / total records in single partition are ~14%. Batch combines different paritions to keep later data coding partitions to be lower in number 

In [ ]:
batch_groups = [(1, 7), (8, 14), (15, 21)]

In [38]:
for beg, end in batch_groups:
    frames = []
    for i in range(beg, end + 1):
        serving_csv = Path(screened_folder) / str(i) / "serving" / "screening_eligible.csv"
        if not serving_csv.exists():
            raise FileNotFoundError(f"Missing serving file for partition {i}: {serving_csv}")
        frames.append(pd.read_csv(serving_csv))

    batch_df = pd.concat(frames, ignore_index=True)

    if len(batch_df) > batched_folder_limit:
        raise ValueError(
            f"Batch {beg}-{end} has {len(batch_df)} records, exceeds limit of {batched_folder_limit}"
        )

    out_dir = Path(batched_folder) / f"{beg}-{end}"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / "screening_eligible.csv"
    batch_df.to_csv(out_path, index=False)

    meta = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "description": f"Batched serving (eligible) records from partitions {beg}-{end}",
        "partitions": list(range(beg, end + 1)),
        "num_records": len(batch_df),
        "limit": batched_folder_limit,
        "columns": list(batch_df.columns),
    }
    (out_dir / "metadata.json").write_text(json.dumps(meta, indent=2))

    print(f"Batch {beg}-{end}: {len(batch_df)} records → {out_path}")

Batch 1-7: 9250 records → data/partitions-screened-batched/1-7/screening_eligible.csv
Batch 8-14: 9469 records → data/partitions-screened-batched/8-14/screening_eligible.csv
Batch 15-21: 9405 records → data/partitions-screened-batched/15-21/screening_eligible.csv


### 2. Data Coding Task

In [5]:
task_name = "driver"

# Run name like pX-X-l{1-6}-ddmmyy-0x
suffix = "l1-250226-01"
# partition_range = [(1, 7), (8, 14), (15, 21)]
partition_range = [(8, 14)]

In [6]:
for x in partition_range:
    _min, _max = x[0], x[1]
    cmd = f"python labelling/orchestrator.py data/partitions-screened-batched/{_min}-{_max} p{_min}-{_max}-{suffix} --task {task_name}"
    print(f"\n{'='*60}")
    print(f"Running partition {x}: {cmd}")
    print(f"{'='*60}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(f"STDERR: {result.stderr}")
    if result.returncode != 0:
        print(f"FAILED with return code {result.returncode}")
        break
    print(f"Partition {x} complete.")


Running partition (8, 14): python labelling/orchestrator.py data/partitions-screened-batched/8-14 p8-14-l1-250226-01 --task driver
Loaded 5 documents from data/partitions-screened-batched/8-14
Dataset saved to /Users/hunain/d/coding-projects/biodiversity/labelling/artifacts/datasets/p8-14-l1-250226-01-dataset.json
Wrote 1 batches to /Users/hunain/d/coding-projects/biodiversity/labelling/artifacts/batches/p8-14-l1-250226-01/data
Prepared 1 request files in /Users/hunain/d/coding-projects/biodiversity/labelling/artifacts/batches/p8-14-l1-250226-01/request
Wrote 5 live outputs to /Users/hunain/d/coding-projects/biodiversity/labelling/artifacts/batch_outputs/p8-14-l1-250226-01/drivers-batch-p8-14-l1-250226-01-001-requests.jsonl (skipped 0)
Submission summary:
  [driver] Live outputs for /Users/hunain/d/coding-projects/biodiversity/labelling/artifacts/batches/p8-14-l1-250226-01/request/batch-p8-14-l1-250226-01-001-requests.jsonl: /Users/hunain/d/coding-projects/biodiversity/labelling/artif

#### Process data coding outputs to /data/partitions-coded

In [7]:
# load source data once for the curated join
_wos_cols = ["UT (Unique WOS ID)", "Article Title", "Abstract", "Publication Year"]
wos_coded_df = pd.read_csv(source_data, usecols=_wos_cols)

for beg, end in partition_range:
    run_name = f"p{beg}-{end}-{suffix}"
    src_dir = Path(batch_outputs_folder) / run_name
    if not src_dir.exists():
        print(f"⚠ Missing: {src_dir}, skipping")
        continue

    # create per-partition folder structure
    part_dir = Path(coded_folder) / task_name / f"{beg}-{end}"
    raw_dir = part_dir / "raw"
    curated_dir = part_dir / "curated"
    for d in [raw_dir, curated_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # ── RAW: concat all jsonl files as-is ─────────────────────────────
    raw_rows = []
    for fp in sorted(src_dir.glob("*.jsonl")):
        with open(fp) as f:
            for line in f:
                rec = json.loads(line)
                rec["_source_run"] = run_name
                rec["_source_file"] = fp.name
                raw_rows.append(rec)

    raw_df = pd.DataFrame(raw_rows)
    raw_path = raw_dir / f"{task_name}_raw.csv"
    raw_df.to_csv(raw_path, index=False)

    raw_meta = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "description": f"Raw batch output JSONL records for {task_name}",
        "partitions": f"{beg}-{end}",
        "run": run_name,
        "task": task_name,
        "num_records": len(raw_df),
        "columns": list(raw_df.columns),
    }
    (raw_dir / "metadata.json").write_text(json.dumps(raw_meta, indent=2))

    # ── CURATED: parse via evals_local + enrich from source_data ──────
    curated_df = load_predictions(task_name, run_name)
    n_before = len(curated_df)

    curated_df = curated_df.merge(
        wos_coded_df,
        left_on="custom_id",
        right_on="UT (Unique WOS ID)",
        how="left",
    ).drop(columns=["UT (Unique WOS ID)"])

    assert len(curated_df) == n_before, (
        f"{beg}-{end}: row count changed after join ({n_before} → {len(curated_df)})"
    )
    unmatched = curated_df["Article Title"].isna().sum()
    if unmatched:
        print(f"  ⚠ {beg}-{end}: {unmatched} records did not match source_data")

    curated_path = curated_dir / f"{task_name}_curated.csv"
    curated_df.to_csv(curated_path, index=False)

    curated_meta = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "description": f"Parsed {task_name} predictions enriched with Article Title, Abstract, Publication Year",
        "partitions": f"{beg}-{end}",
        "run": run_name,
        "task": task_name,
        "num_records": len(curated_df),
        "enriched_from": source_data,
        "enriched_fields": ["Article Title", "Abstract", "Publication Year"],
        "columns": list(curated_df.columns),
    }
    (curated_dir / "metadata.json").write_text(json.dumps(curated_meta, indent=2))

    print(f"{task_name} {beg}-{end}: raw={len(raw_df)}  curated={len(curated_df)} → {part_dir}")

driver 8-14: raw=5  curated=5 → data/partitions-coded/driver/8-14
